### Bike Demand Forecasting Model

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('data/SeoulBikeData.csv', encoding="cp949")
df.head()

,Date,Rented Bike Count,Hour,Temperature(캜),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(캜),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [4]:
# 컬럼명 수정

df.columns = [
    "date",
    "bike_count",
    "hour",
    "temperature",
    "humidity",
    "wind_speed",
    "visibility",
    "dew_point",
    "solar_radiation",
    "rainfall",
    "snowfall",
    "season",
    "holiday",
    "functioning_day"
]
df.head()

,date,bike_count,hour,temperature,humidity,wind_speed,visibility,dew_point,solar_radiation,rainfall,snowfall,season,holiday,functioning_day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [5]:
print(df.shape)
print(df.info())

(8760, 14)
<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             8760 non-null   str    
 1   bike_count       8760 non-null   int64  
 2   hour             8760 non-null   int64  
 3   temperature      8760 non-null   float64
 4   humidity         8760 non-null   int64  
 5   wind_speed       8760 non-null   float64
 6   visibility       8760 non-null   int64  
 7   dew_point        8760 non-null   float64
 8   solar_radiation  8760 non-null   float64
 9   rainfall         8760 non-null   float64
 10  snowfall         8760 non-null   float64
 11  season           8760 non-null   str    
 12  holiday          8760 non-null   str    
 13  functioning_day  8760 non-null   str    
dtypes: float64(6), int64(4), str(4)
memory usage: 958.3 KB
None


In [6]:
# 중복데이터 확인
df.duplicated().sum()

np.int64(0)

In [7]:
df.describe()

,bike_count,hour,temperature,humidity,wind_speed,visibility,dew_point,solar_radiation,rainfall,snowfall
count,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000
mean,704.602055,11.500000,12.882922,58.226256,1.724909,1436.825799,4.073813,0.569111,0.148687,0.075068
std,644.997468,6.922582,11.944825,20.362413,1.036300,608.298712,13.060369,0.868746,1.128193,0.436746
min,0.000000,0.000000,-17.800000,0.000000,0.000000,27.000000,-30.600000,0.000000,0.000000,0.000000
25%,191.000000,5.750000,3.500000,42.000000,0.900000,940.000000,-4.700000,0.000000,0.000000,0.000000
50%,504.500000,11.500000,13.700000,57.000000,1.500000,1698.000000,5.100000,0.010000,0.000000,0.000000
75%,1065.250000,17.250000,22.500000,74.000000,2.300000,2000.000000,14.800000,0.930000,0.000000,0.000000
max,3556.000000,23.000000,39.400000,98.000000,7.400000,2000.000000,27.200000,3.520000,35.000000,8.800000


In [8]:
# 1. date 컬럼을 datetime 타입으로 변환
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

# 2. 날짜 기반 유용한 파생 피처 추출
df['month'] = df['date'].dt.month
df['dayofweek'] = df['date'].dt.dayofweek
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

# 3. 원본 date 컬럼 및 기타 범주형 텍스트 처리
# (season, holiday, functioning_day 등 텍스트 컬럼은 원-핫 인코딩 필요)
df = df.drop(columns=['date'])

df.head()

,bike_count,hour,temperature,humidity,wind_speed,visibility,dew_point,solar_radiation,rainfall,snowfall,season,holiday,functioning_day,month,dayofweek,is_weekend
0,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,12,4,0
1,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,12,4,0
2,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes,12,4,0
3,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,12,4,0
4,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes,12,4,0


In [9]:
# 사용 안할 feature : visibility, dew_point, solar_radiation, snowfall, season, functioning_day

df = df.drop(columns=['visibility', 'dew_point', 'solar_radiation', 'snowfall', 'season', 'functioning_day'])

df.head()

,bike_count,hour,temperature,humidity,wind_speed,rainfall,holiday,month,dayofweek,is_weekend
0,254,0,-5.2,37,2.2,0.0,No Holiday,12,4,0
1,204,1,-5.5,38,0.8,0.0,No Holiday,12,4,0
2,173,2,-6.0,39,1.0,0.0,No Holiday,12,4,0
3,107,3,-6.2,40,0.9,0.0,No Holiday,12,4,0
4,78,4,-6.0,36,2.3,0.0,No Holiday,12,4,0


In [10]:
# 'No Holiday'가 아니면 1, 'No Holiday'이면 0으로 변환
df['holiday'] = (df['holiday'] != 'No Holiday').astype(int)

In [11]:
df['holiday'].value_counts()

holiday
0    8328
1     432
Name: count, dtype: int64

In [12]:
df.head()

,bike_count,hour,temperature,humidity,wind_speed,rainfall,holiday,month,dayofweek,is_weekend
0,254,0,-5.2,37,2.2,0.0,0,12,4,0
1,204,1,-5.5,38,0.8,0.0,0,12,4,0
2,173,2,-6.0,39,1.0,0.0,0,12,4,0
3,107,3,-6.2,40,0.9,0.0,0,12,4,0
4,78,4,-6.0,36,2.3,0.0,0,12,4,0


In [17]:
import os
import pandas as pd
import joblib

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [14]:
X = df.drop(columns=["bike_count"])
y = df["bike_count"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [18]:
# 다이어그램 출력 끄기 설정
sklearn.set_config(display='text')

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)  # 학습

RandomForestRegressor(random_state=42)

In [19]:
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("MAE:", round(mae, 2))
print("R2 Score:", round(r2, 4))

MAE: 137.14
R2 Score: 0.8368


In [20]:
joblib.dump(
    model,
    "model/bike_demand_model.pkl"
)

print("모델 저장 완료: model/bike_demand_model.pkl")

모델 저장 완료: model/bike_demand_model.pkl


In [21]:
X.head()

,hour,temperature,humidity,wind_speed,rainfall,holiday,month,dayofweek,is_weekend
0,0,-5.2,37,2.2,0.0,0,12,4,0
1,1,-5.5,38,0.8,0.0,0,12,4,0
2,2,-6.0,39,1.0,0.0,0,12,4,0
3,3,-6.2,40,0.9,0.0,0,12,4,0
4,4,-6.0,36,2.3,0.0,0,12,4,0
